In [1]:
import sqlite3
import csv
import urllib.request
import os

In [2]:
# GitHub raw URLs (we’ll fix if needed)
BASE_URL = "https://raw.githubusercontent.com/<username>/<repo>/main/"

BOOK_URL   = BASE_URL + "Book.csv"
MEMBER_URL = BASE_URL + "Member.csv"
LOAN_URL   = BASE_URL + "Loan.csv"

# Local paths
BOOK_PATH   = "/content/Book.csv"
MEMBER_PATH = "/content/Member.csv"
LOAN_PATH   = "/content/Loan.csv"

DB_PATH = "/content/library.db"

print("Paths set")

Paths set


In [3]:
from google.colab import files
uploaded = files.upload()

Saving Loan.csv to Loan.csv
Saving Member.csv to Member.csv
Saving Book.csv to Book.csv


In [4]:
BOOK_PATH = "Book.csv"
MEMBER_PATH = "Member.csv"
LOAN_PATH = "Loan.csv"

DB_PATH = "library.db"

print("Files ready")

Files ready


In [5]:
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")

conn.execute("""
CREATE TABLE IF NOT EXISTS Book (
    callNo  TEXT    NOT NULL,
    title   TEXT    NOT NULL,
    author  TEXT    NOT NULL,
    PRIMARY KEY (callNo)
);
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS Member (
    id          INTEGER NOT NULL,
    firstname   TEXT    NOT NULL,
    lastName    TEXT    NOT NULL,
    PRIMARY KEY (id)
);
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS Loan (
    callNo        TEXT    NOT NULL,
    id            INTEGER NOT NULL,
    dateBorrowed  TEXT    NOT NULL,
    dateReturned  TEXT,
    dateDue       TEXT    NOT NULL,
    PRIMARY KEY (callNo, id, dateBorrowed),
    FOREIGN KEY (callNo) REFERENCES Book(callNo),
    FOREIGN KEY (id) REFERENCES Member(id)
);
""")

conn.commit()

print("Tables created.")
print(conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;").fetchall())

Tables created.
[('Book',), ('Loan',), ('Member',)]


In [6]:
with open(BOOK_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            "INSERT INTO Book (callNo, title, author) VALUES (?, ?, ?);",
            (row["callNo"], row["title"], row["author"])
        )

conn.commit()
print("Book rows loaded:", conn.execute("SELECT COUNT(*) FROM Book;").fetchone()[0])

Book rows loaded: 11


In [7]:
with open(MEMBER_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            "INSERT INTO Member (id, firstname, lastName) VALUES (?, ?, ?);",
            (int(row["id"]), row["firstname"], row["lastName"])
        )

conn.commit()
print("Member rows loaded:", conn.execute("SELECT COUNT(*) FROM Member;").fetchone()[0])

Member rows loaded: 4


In [8]:
with open(LOAN_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        date_returned = row["dateReturned"] if row["dateReturned"].strip() else None

        conn.execute(
            """
            INSERT INTO Loan (callNo, id, dateBorrowed, dateReturned, dateDue)
            VALUES (?, ?, ?, ?, ?);
            """,
            (
                row["callNo"],
                int(row["id"]),
                row["dateBorrowed"],
                date_returned,
                row["dateDue"]
            )
        )

conn.commit()
print("Loan rows loaded:", conn.execute("SELECT COUNT(*) FROM Loan;").fetchone()[0])

Loan rows loaded: 4


In [9]:
query1 = """
SELECT *
FROM Book
ORDER BY author;
"""

rows = conn.execute(query1).fetchall()
for row in rows:
    print(row)

('R 487 T35 1967', 'Medicine in medieval England.', 'Charles H Talbot')
('QA 76.9 D26H39 1996', 'Data model patterns : conventions of thought', 'David Hay')
('CB 351 M293 1983', 'Atlas of medieval Europe', 'Donald Matthew')
('HQ 1143 P68 1975', 'Medieval women', 'Eileen Power')
('PC 14 V48 1965', 'Medieval miscellany', 'Frederick Whitehead')
('QA 76.73 S67C435 2004', "Joe Celko's Trees and hierarchies in SQL for smarties", 'Joe Celko')
('QA 76.73 S67C46 1997', "Joe Celko's SQL puzzles & answers", 'Joe Celko')
('QA 76.9 D35C45 1999', "Joe Celko's data & databases : concepts in practice", 'Joe Celko')
('R 141 E45 2006', 'Medieval medicine and the plague', 'Lynne Elliott')
('QA 76.9 D26H355 2008', 'Information modeling and relational databases', 'T A Halpin')
('QA 76.76 A65P76 2011', 'Programming Android', 'Zigurd R Mednieks')


In [10]:
query2 = """
SELECT Book.title, Member.firstname, Member.lastName
FROM Loan
JOIN Book ON Loan.callNo = Book.callNo
JOIN Member ON Loan.id = Member.id
WHERE Loan.dateReturned IS NULL;
"""

rows = conn.execute(query2).fetchall()
for row in rows:
    print(row)

("Joe Celko's SQL puzzles & answers", 'David', 'Martin')
('Medieval medicine and the plague', 'David', 'Martin')


In [11]:
query3 = """
SELECT Member.firstname, Member.lastName, Loan.dateBorrowed, Loan.dateDue, Loan.dateReturned
FROM Loan
JOIN Member ON Loan.id = Member.id
WHERE Loan.callNo = 'R 141 E45 2006'
ORDER BY Loan.dateBorrowed ASC;
"""

rows = conn.execute(query3).fetchall()
for row in rows:
    print(row)

('Betty', 'Freeman', '4/1/2014 0:00', '4/15/2014 0:00', '4/15/2014 0:00')
('David', 'Martin', '4/30/2014 0:00', '5/14/2014 0:00', None)


In [12]:
query4 = """
SELECT Member.id, Member.firstname, Member.lastName
FROM Member
LEFT JOIN Loan ON Member.id = Loan.id
WHERE Loan.id IS NULL;
"""

rows = conn.execute(query4).fetchall()
for row in rows:
    print(row)

(4, 'John', 'Martin')


In [13]:
query5 = """
SELECT Member.firstname, Member.lastName, COUNT(Loan.callNo) AS total_loans
FROM Member
LEFT JOIN Loan ON Member.id = Loan.id
GROUP BY Member.id, Member.firstname, Member.lastName
ORDER BY total_loans DESC;
"""

rows = conn.execute(query5).fetchall()
for row in rows:
    print(row)

('David', 'Martin', 2)
('John', 'Smith', 1)
('Betty', 'Freeman', 1)
('John', 'Martin', 0)


In [16]:
query6 = """
SELECT Book.author, COUNT(*) AS total_loans
FROM Loan
JOIN Book ON Loan.callNo = Book.callNo
GROUP BY Book.author
ORDER BY total_loans DESC;
"""

rows = conn.execute(query6).fetchall()
for row in rows:
    print(row)

('Lynne Elliott', 2)
('Joe Celko', 1)
('Eileen Power', 1)


# Summary

One data quality issue I noticed is that the `dateReturned` column contains blank values for books that have not yet been returned. These blank values had to be converted into `NULL` in SQLite so the queries could correctly identify outstanding loans. Another issue is that the date fields are stored as text, which works for this assignment but would make more advanced date calculations harder in a real system. One limitation of this dataset is that it is very small and only includes a few books, members, and loan records. A real public library database would likely include more details such as multiple copies of the same book, late fees, reservations, and book categories.

In [17]:
conn.close()
print("Database connection closed.")

Database connection closed.
